In [63]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

from keras.models import Model
from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

In [64]:
CSV_PATH = "../data/ISIC_2020_Training_GroundTruth.csv"

TRAIN_DIR = "../data/images/train"
VAL_DIR = "../data/images/val"
TEST_DIR = "../data/images/test"

IMG_HEIGHT = 512
IMG_WIDTH = 512
BATCH_SIZE = 32
EPOCHS = 10

In [65]:
# Load ground truth data
df = pd.read_csv(CSV_PATH)
df.head()

,image_name,patient_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target
0,ISIC_2637011,IP_7279968,male,45.0,head/neck,unknown,benign,0
1,ISIC_0015719,IP_3075186,female,45.0,upper extremity,unknown,benign,0
2,ISIC_0052212,IP_2842074,female,50.0,lower extremity,nevus,benign,0
3,ISIC_0068279,IP_6890425,female,45.0,head/neck,unknown,benign,0
4,ISIC_0074268,IP_8723313,female,55.0,upper extremity,unknown,benign,0


In [66]:
df = df[["image_name", "target"]]
df.head()

,image_name,target
0,ISIC_2637011,0
1,ISIC_0015719,0
2,ISIC_0052212,0
3,ISIC_0068279,0
4,ISIC_0074268,0


In [67]:
# Check class imbalance
labels_df["target"].value_counts().sort_index()

target
0    32542
1      584
Name: count, dtype: int64

In [68]:
# Label lookup dictionary
label_dict = dict(zip(df["image_name"], df["target"]))

In [69]:
# Build df for training
train_files = []
train_labels = []

for file in os.listdir(TRAIN_DIR):
    if file.endswith(".jpg"):
        image_name = file.replace(".jpg", "")
        
        if image_name in label_dict:
            train_files.append(os.path.join(TRAIN_DIR, file))
            train_labels.append(label_dict[image_name])

train_df = pd.DataFrame({
    "filepath": train_files,
    "label": train_labels
})

train_df.head()

,filepath,label
0,../data/images/train/ISIC_6177679.jpg,0
1,../data/images/train/ISIC_7979115.jpg,0
2,../data/images/train/ISIC_8869597.jpg,0
3,../data/images/train/ISIC_1507333.jpg,0
4,../data/images/train/ISIC_7097665.jpg,0


In [70]:
# Compute class weights
classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].values
)

class_weight = {
    0: weights[0],
    1: weights[1]
}

print(class_weight)

{0: np.float64(0.5079051383399209), 1: np.float64(32.125)}


In [71]:
# Build df for validation
val_files = []
val_labels = []

for file in os.listdir(VAL_DIR):
    if file.endswith(".jpg"):
        image_name = file.replace(".jpg", "")
        
        if image_name in label_dict:
            val_files.append(os.path.join(VAL_DIR, file))
            val_labels.append(label_dict[image_name])

val_df = pd.DataFrame({
    "filepath": val_files,
    "label": val_labels
})

val_df.head()

,filepath,label
0,../data/images/val/ISIC_2360676.jpg,0
1,../data/images/val/ISIC_8756356.jpg,0
2,../data/images/val/ISIC_6126620.jpg,0
3,../data/images/val/ISIC_9115484.jpg,0
4,../data/images/val/ISIC_2904907.jpg,0


In [72]:
# Build df for testing
test_files = []
test_labels = []

for file in os.listdir(TEST_DIR):
    if file.endswith(".jpg"):
        image_name = file.replace(".jpg", "")
        
        if image_name in label_dict:
            test_files.append(os.path.join(TEST_DIR, file))
            test_labels.append(label_dict[image_name])

test_df = pd.DataFrame({
    "filepath": test_files,
    "label": test_labels
})

test_df.head()

,filepath,label
0,../data/images/test/ISIC_2645615.jpg,0
1,../data/images/test/ISIC_6793983.jpg,0
2,../data/images/test/ISIC_5370367.jpg,0
3,../data/images/test/ISIC_1299998.jpg,0
4,../data/images/test/ISIC_0558046.jpg,0


In [73]:
# Check class imbalance
print("Train:")
print(train_df["label"].value_counts())

print("\nValidation:")
print(val_df["label"].value_counts())

print("\nTest:")
print(test_df["label"].value_counts())

Train:
label
0    253
1      4
Name: count, dtype: int64

Validation:
label
0    238
1      2
Name: count, dtype: int64

Test:
label
0    593
1      7
Name: count, dtype: int64


In [74]:
def load_image(filepath, label):
    image = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = image / 255.0
    return image, label

In [75]:
# Create train, validation, test datasets
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["filepath"].values, train_df["label"].values)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["filepath"].values, val_df["label"].values)
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_df["filepath"].values, test_df["label"].values)
)

train_ds = train_ds.map(load_image).shuffle(len(train_df)).batch(BATCH_SIZE)
val_ds = val_ds.map(load_image).batch(BATCH_SIZE)
test_ds = test_ds.map(load_image).batch(BATCH_SIZE)

In [76]:
inputs = Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))

x = Conv2D(32, (3, 3), activation="relu")(inputs)
x = MaxPooling2D((2, 2))(x)

x = Conv2D(64, (3, 3), activation="relu")(x)
x = MaxPooling2D((2, 2))(x)

x = Conv2D(128, (3, 3), activation="relu")(x)
x = MaxPooling2D((2, 2))(x)

x = Conv2D(256, (3, 3), activation="relu")(x)
x = MaxPooling2D((2, 2))(x)

x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)

output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=inputs, outputs=output)

In [77]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

In [78]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight
)

Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 28s 3s/step - accuracy: 0.8827 - loss: 2.6482 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9917 - val_loss: 0.0496 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 34s 4s/step - accuracy: 0.9808 - loss: 2.3985 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9917 - val_loss: 0.5645 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 37s 4s/step - accuracy: 0.9833 - loss: 0.7032 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9917 - val_loss: 0.6439 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 30s 3s/step - accuracy: 0.9838 - loss: 0.6804 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9917 - val_loss: 0.6583 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 30s 3s/step - accuracy: 0.9894 - loss: 0.5771 - precision: 0.0000e+00 - reca

In [79]:
results = model.evaluate(test_ds)
print(results)
print(model.metrics_names)

19/19 ━━━━━━━━━━━━━━━━━━━━ 14s 729ms/step - accuracy: 0.9874 - loss: 0.5960 - precision: 0.0000e+00 - recall: 0.0000e+00
[0.5956495404243469, 0.9883333444595337, 0.0, 0.0]
['loss', 'compile_metrics']


In [80]:
y_prob = model.predict(test_ds)

for threshold in [0.5, 0.4, 0.3, 0.2]:
    y_pred = (y_prob > threshold).astype(int).flatten()
    print(f"\nThreshold = {threshold}")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred))

19/19 ━━━━━━━━━━━━━━━━━━━━ 14s 704ms/step

Threshold = 0.5
[[593   0]
 [  7   0]]
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       593
           1       0.00      0.00      0.00         7

    accuracy                           0.99       600
   macro avg       0.49      0.50      0.50       600
weighted avg       0.98      0.99      0.98       600


Threshold = 0.4
[[ 53 540]
 [  0   7]]
              precision    recall  f1-score   support

           0       1.00      0.09      0.16       593
           1       0.01      1.00      0.03         7

    accuracy                           0.10       600
   macro avg       0.51      0.54      0.09       600
weighted avg       0.99      0.10      0.16       600


Threshold = 0.3
[[  4 589]
 [  0   7]]
              precision    recall  f1-score   support

           0       1.00      0.01      0.01       593
           1       0.01      1.00      0.02         7

    accuracy        

/Users/yaroslawbagriy/Dev/venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/yaroslawbagriy/Dev/venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/yaroslawbagriy/Dev/venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(r

In [81]:
y_true = test_df["label"].values

In [82]:
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Confusion Matrix:
[[  0 593]
 [  0   7]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       593
           1       0.01      1.00      0.02         7

    accuracy                           0.01       600
   macro avg       0.01      0.50      0.01       600
weighted avg       0.00      0.01      0.00       600



/Users/yaroslawbagriy/Dev/venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/yaroslawbagriy/Dev/venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/yaroslawbagriy/Dev/venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(r

In [83]:
os.makedirs("../models", exist_ok=True)
model.save("../models/basic_melanoma_binary_classifier.keras")

# 1. Results of basic classifier
```
Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       593
           1       0.00      0.00      0.00         7

    accuracy                           0.99       600
   macro avg       0.49      0.50      0.50       600
weighted avg       0.98      0.99      0.98       600
```

# 2. Results of basic classifier with improvments
- Compute class weights after train_df is created and use them for training
- Use precision and recall as metrics for training
- Find best threshold
- Make smaller model so it doens't overfit too quickly
- Increase training image size to include more resolution (512x512)

```
Threshold = 0.4
              precision    recall  f1-score   support

           0       1.00      0.09      0.16       593
           1       0.01      1.00      0.03         7

    accuracy                           0.10       600
   macro avg       0.51      0.54      0.09       600
weighted avg       0.99      0.10      0.16       600
```